# 🏥 Medical Report Analysis System — Google Colab Runner

This notebook configures and runs the full 3-model medical pipeline (**LLaMA 3 8B → Meditron 7B → BioMistral 7B**) with GPU acceleration, and provides a full framework to **LoRA Fine-Tune** the model on Colab.

### ⚙️ Google Colab Requirements
- **Runtime**: You **MUST** use a GPU runtime (T4, L4, or A100). Go to `Runtime > Change runtime type` and select **GPU**.
- **Kaggle API Key**: Have your `kaggle.json` file ready to upload for the dataset downloads.

## 🛠️ Step 1: Environment Setup & GPU Installation
We compile `llama-cpp-python` with **CUDA support** to offload model execution to the Colab GPU, making inference extremely fast. We also install HuggingFace training dependencies like `peft`, `trl`, and `accelerate`.

In [ ]:
# 1. Install GPU-accelerated llama-cpp-python (this takes 2-4 minutes to compile with CUDA)
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python==0.3.19 --no-cache-dir

# 2. Install all other project dependencies (including LoRA training tools)
!pip install pymupdf pytesseract pillow paddleocr gradio fpdf2 python-dotenv regex datasets kaggle huggingface_hub pandas tqdm transformers peft trl accelerate bitsandbytes

## 📂 Step 2: Upload credentials (`kaggle.json`)
Run the cell below and upload the `kaggle.json` file you have in the project folder to enable Kaggle medical dataset downloads.

In [ ]:
from google.colab import files
import os
import shutil

uploaded = files.upload()
if "kaggle.json" in uploaded:
    # Place in current directory for download_datasets.py to auto-load
    print("\n[+] Credentials loaded successfully!")
else:
    print("\n[!] Please upload kaggle.json to download the Kaggle datasets.")

## 📥 Step 3: Run Model and Dataset Downloads

In [ ]:
# Download the 3 GGUF models (LLaMA 3, Meditron, BioMistral) for the dashboard
!python download_models.py

In [ ]:
# Download the 6 training datasets (PubMedQA, MedMCQA, MedQA, ChatDoctor, HealthCareMagic, MedMNIST)
!python download_datasets.py

## 🚀 Step 4: Launch Web Dashboard
Run this cell to start the Gradio web dashboard. It will automatically detect Colab and generate a public **`https://*.gradio.live`** URL. Open that URL in your browser to upload reports and run analyses.

In [ ]:
# Run Gradio interface
!python app/main.py

## 🧠 Step 5: How to Fine-Tune via QLoRA / LoRA (Optional)

To customize/fine-tune the BioMistral model on the medical datasets you downloaded, follow these sub-steps:

### A. Prepare Training Dataset
This compiles instructions, Q&As, and doctor dialogues into standard instruction-following format.

In [ ]:
!python fine_tuning/dataset_prep.py

### B. Run LoRA Training Session
Trains adapters using **QLoRA (4-bit quantization)** to fit inside 16GB VRAM. This is configured to run for 1 epoch on the subset of data.

In [ ]:
!python fine_tuning/train_lora.py

### C. Merge PEFT Adapters Back into Base Model
Consolidates the base model and your newly trained adapters into a single fine-tuned model directory.

In [ ]:
!python fine_tuning/merge_lora.py